# Kaggle Notebook: Obesity Risk Classification

This notebook is designed to run **locally** with the files `train.csv` and `test.csv` stored in the **same folder** as this notebook.

## What this notebook does
- Loads the Kaggle training and test data
- Performs light exploratory review
- Builds a preprocessing pipeline for numeric and categorical features
- Compares multiple classification models
- Selects the best model based on validation accuracy
- Generates a `submission.csv` file for Kaggle upload

## Local folder expectation
Keep these three files together:
- `Kaggle_Obesity_Classification_Notebook_v2.ipynb`
- `train.csv`
- `test.csv`


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


In [ ]:
# Load the local CSV files from the same folder as this notebook
base_path = Path(".")
train_path = base_path / "train.csv"
test_path = base_path / "test.csv"

if not train_path.exists():
    raise FileNotFoundError("train.csv was not found in the same folder as this notebook.")

if not test_path.exists():
    raise FileNotFoundError("test.csv was not found in the same folder as this notebook.")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


In [ ]:
# Quick inspection
display(train_df.head())
display(test_df.head())

print("Training columns:")
display(train_df.columns.tolist())

print("Test columns:")
display(test_df.columns.tolist())


In [ ]:
# Basic data quality review
missing_summary = pd.DataFrame({
    "train_missing": train_df.isna().sum(),
    "test_missing": test_df.isna().sum().reindex(train_df.columns, fill_value=np.nan)
})

display(missing_summary)

print("Target distribution:")
display(train_df["NObeyesdad"].value_counts().sort_index())


In [ ]:
# Simple visual review
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

train_df["NObeyesdad"].value_counts().sort_values().plot(kind="barh", ax=axes[0], title="Target Class Distribution")
train_df["Age"].plot(kind="hist", bins=30, ax=axes[1], title="Age Distribution")

plt.tight_layout()
plt.show()


## Prepare Features and Target

In [ ]:
target_col = "NObeyesdad"
id_col = "id"

X = train_df.drop(columns=[target_col])
y = train_df[target_col]
X_test = test_df.copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

categorical_features = X_train.select_dtypes(include="object").columns.tolist()
numeric_features = [col for col in X_train.columns if col not in categorical_features]

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)


## Build Preprocessing Pipeline

The notebook uses:
- median imputation + scaling for numeric variables
- most-frequent imputation + one-hot encoding for categorical variables


In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


## Train and Compare Candidate Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=3000),
    "Random Forest": RandomForestClassifier(
        n_estimators=400,
        random_state=42,
        n_jobs=-1
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    ),
}

results = []
fitted_models = {}

for model_name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    valid_preds = pipeline.predict(X_valid)
    valid_accuracy = accuracy_score(y_valid, valid_preds)

    results.append({
        "model": model_name,
        "validation_accuracy": valid_accuracy
    })
    fitted_models[model_name] = pipeline

results_df = pd.DataFrame(results).sort_values(
    by="validation_accuracy",
    ascending=False
).reset_index(drop=True)

display(results_df)


## Select the Best Model and Evaluate It

In [ ]:
best_model_name = results_df.loc[0, "model"]
best_pipeline = fitted_models[best_model_name]

print("Best model:", best_model_name)

best_preds = best_pipeline.predict(X_valid)

print("\nClassification Report:\n")
print(classification_report(y_valid, best_preds))


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_valid,
    best_preds,
    xticks_rotation=45
)
plt.title(f"Confusion Matrix: {best_model_name}")
plt.tight_layout()
plt.show()


## Retrain the Best Model on All Training Data and Create Kaggle Submission

In [ ]:
best_pipeline.fit(X, y)
test_predictions = best_pipeline.predict(X_test)

submission_df = pd.DataFrame({
    "id": test_df[id_col],
    "NObeyesdad": test_predictions
})

submission_path = base_path / "submission.csv"
submission_df.to_csv(submission_path, index=False)

print("Saved submission file to:", submission_path.resolve())
display(submission_df.head())


## Interpretation Notes

A few things to mention in your written discussion:

- The validation comparison shows how different classifiers perform on the same preprocessed feature set.
- The best model is selected strictly by validation accuracy, which keeps the choice transparent and reproducible.
- The final submission is produced only after retraining the best-performing model on the full labeled training data.
- Because the target is multiclass, the classification report is useful for checking whether the model performs consistently across all obesity classes rather than relying only on one summary metric.
